# 🚀 DeBERTa Resume NER Training - Final Fixed Version
## Complete training pipeline

**Labels:** PERSON_NAME, COMPANY, CLIENT, ROLE, LOCATION, DATE_START, DATE_END, DEGREE, FIELD, INSTITUTION, EDU_YEAR_START, EDU_YEAR_END, GRADE

**Dataset:** 17,363 samples (15,433 train, 1,930 test)

In [ ]:
# ============================================================================
# STEP 1: VERIFY GPU
# ============================================================================
import torch
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("❌ NO GPU - STOP HERE")
    raise SystemExit("GPU not enabled. Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
# ============================================================================
# STEP 2: UPLOAD AND EXTRACT
# ============================================================================
import os
from google.colab import files

# Clean workspace
!rm -rf /content/Lakshya-Colab-Training /content/*.zip
print("🧹 Cleaned workspace\n")

# Change to /content directory first
os.chdir('/content')

# Upload ZIP
print("📤 Upload your Lakshya-Colab-Training.zip file...")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

# Extract
!unzip -q "{zip_name}" -d /content/
print(f"\n✅ Extracted {zip_name}")

# Navigate to training directory
%cd /content/Lakshya-Colab-Training/ai-service
!pwd

In [ ]:
# ============================================================================
# STEP 3: INSTALL DEPENDENCIES
# ============================================================================
print("📦 Installing dependencies...\n")
!pip install -q transformers==4.44.0 datasets==2.19.0 accelerate==0.33.0 evaluate==0.4.1 seqeval scikit-learn torch==2.4.0
print("\n✅ Dependencies installed")

In [ ]:
# ============================================================================
# STEP 4: START TRAINING
# ============================================================================
print("\n🚀 Starting DeBERTa training...")
print("⏱️  Expected time: ~30-40 minutes on T4 GPU")
print("="*60)
!python training/train_colab_standalone.py

In [ ]:
# ============================================================================
# STEP 5: SAVE TO GOOGLE DRIVE
# ============================================================================
from google.colab import drive

print("💾 Mounting Google Drive...")
drive.mount('/content/drive')

# Copy to Drive
!mkdir -p "/content/drive/MyDrive/Resume-NER-Models"
!cp -r /content/Lakshya-Colab-Training/ai-service/models/resume-ner-deberta "/content/drive/MyDrive/Resume-NER-Models/"
print("\n✅ Model saved to Google Drive: /MyDrive/Resume-NER-Models/resume-ner-deberta")

# Verify
print("\n📂 Model files:")
!ls -lh "/content/drive/MyDrive/Resume-NER-Models/resume-ner-deberta"

In [ ]:
# ============================================================================
# STEP 6: DOWNLOAD MODEL ZIP
# ============================================================================
from google.colab import files

print("📦 Creating downloadable ZIP...")

# Create ZIP with full paths
!cd /content/Lakshya-Colab-Training/ai-service && zip -r -q /content/resume-ner-deberta.zip models/resume-ner-deberta

# Verify ZIP was created
!ls -lh /content/resume-ner-deberta.zip

print("\n" + "="*60)
print("🎉 TRAINING COMPLETE!")
print("="*60)
print("\n✅ Model saved to:")
print("   1. Google Drive: /MyDrive/Resume-NER-Models/resume-ner-deberta")
print("   2. Download ZIP: Click below to download")
print("\n📥 Downloading model ZIP...")
files.download('/content/resume-ner-deberta.zip')
print("\n✅ Extract and place in: ai-service/models/resume-ner-deberta")

## 🧪 Test the Model (Optional)
Run this cell to test your trained model

In [ ]:
from transformers import pipeline

# Load the trained model
ner_pipeline = pipeline(
    "ner",
    model="/content/Lakshya-Colab-Training/ai-service/models/resume-ner-deberta",
    aggregation_strategy="simple"
)

# Test with sample text
test_text = """John Smith worked at Infosys as Senior Data Engineer in Hyderabad from Jan 2021 to Mar 2023. 
Client was Google. Education: B.Tech in Computer Science from JNTU Hyderabad, 2015-2019, Grade 8.2"""

print("📝 Test Input:")
print(test_text)
print("\n🔍 Predictions:")
print("="*60)

predictions = ner_pipeline(test_text)
for pred in predictions:
    print(f"{pred['entity_group']:20s} | {pred['word']:30s} | Score: {pred['score']:.3f}")

print("\n✅ Model test complete!")